# Week 6: Spark Architecture & Data Processing Assignment

This notebook covers Spark core architecture, Lazy Evaluation, schema inference, DataFrame transformations, line-of-business filtering, file format comparisons (CSV vs. Parquet), and optimal data pipeline patterns using PySpark.

---


In [1]:
import os
import sys
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Spark_Assignment_Week6") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session initialized successfully.")
print("PySpark Version:", spark.version)

Spark Session initialized successfully.
PySpark Version: 4.2.0


## Q1: Spark Core Components (Driver, Cluster Manager, Executor)

### Explanation
In a Spark application, compute responsibilities are divided among three primary components:

1. **Driver Process**:
   - The Driver is the control node that runs the application's `main()` program and creates the `SparkSession` / `SparkContext`.
   - It parses user code, translates transformations into a Directed Acyclic Graph (DAG) execution plan, splits the DAG into stages and tasks, and schedules tasks across worker nodes.
   - It maintains runtime metadata about application state and collects task metrics.

2. **Cluster Manager**:
   - The Cluster Manager (e.g., YARN, Kubernetes, Mesos, or Spark Standalone) is responsible for physical resource allocation across the cluster.
   - It allocates worker node containers/executors based on application resource requests (CPU cores and RAM) and monitors cluster node health.

3. **Executors**:
   - Executors are worker processes launched on cluster nodes that run individual data processing tasks.
   - They execute task code concurrently across partitions, store cached datasets in memory or on disk, and return task status and results back to the Driver.


In [2]:
sc = spark.sparkContext
print("App Name:", sc.appName)
print("Master Mode:", sc.master)
print("Spark App ID:", sc.applicationId)

App Name: Spark_Assignment_Week6
Master Mode: local[*]
Spark App ID: local-1785052557019


## Q2: Performance Optimization via Lazy Evaluation

### Explanation
**Lazy Evaluation** means Spark does not immediately compute results when DataFrame transformations (`.filter()`, `.select()`, `.withColumn()`, `.groupBy()`) are declared. Instead, it records transformations as an internal Directed Acyclic Graph (DAG) logical execution plan.

Execution is triggered only when an **Action** (`.show()`, `.count()`, `.collect()`, `.write()`) is called.

### Why It Improves Performance
1. **Query Optimization (Catalyst Optimizer)**: Spark evaluates the entire pipeline as a whole before running code. It merges adjacent transformations, eliminates redundant calculations, and applies projection pruning (loading only required columns).
2. **Predicate Pushdown**: Filter operations are pushed directly down to file readers (e.g., Parquet readers), allowing Spark to skip reading non-matching disk blocks into memory.
3. **Pipelining**: Multiple narrow operations pass through memory in a single streaming pass without creating expensive intermediate disk files between steps.


In [3]:
df_raw = spark.read.csv("data/source.csv", header=True, inferSchema=True)
transformed_plan = df_raw.filter(F.col("category") == "Electronics").select("product_id", "price")

print("Logical Plan registered lazily (no execution triggered yet):")
transformed_plan.explain(True)

Logical Plan registered lazily (no execution triggered yet):
== Parsed Logical Plan ==
'Project ['product_id, 'price]
+- Filter (category#18 = Electronics)
   +- Relation [product_id#17,category#18,price#19,old_name#20,status#21,amount#22,base_price#23,region#24,priority#25,user_id#26] csv

== Analyzed Logical Plan ==
product_id: string, price: double
Project [product_id#17, price#19]
+- Filter (category#18 = Electronics)
   +- Relation [product_id#17,category#18,price#19,old_name#20,status#21,amount#22,base_price#23,region#24,priority#25,user_id#26] csv

== Optimized Logical Plan ==
Project [product_id#17, price#19]
+- Filter (isnotnull(category#18) AND (category#18 = Electronics))
   +- Relation [product_id#17,category#18,price#19,old_name#20,status#21,amount#22,base_price#23,region#24,priority#25,user_id#26] csv

== Physical Plan ==
*(1) Project [product_id#17, price#19]
+- *(1) Filter (isnotnull(category#18) AND (category#18 = Electronics))
   +- FileScan csv [product_id#17,categor

## Q3: Reading CSV with Header and Schema Inference

### Question
Write a Spark command to read a CSV file located at `"data/source.csv"`, ensuring the first row is treated as a header and `inferSchema` is enabled.

### Implementation


In [4]:
df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("data/source.csv")

df_csv.printSchema()
df_csv.show(5)

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------+-----------+------+------------+---------+------+----------+------+--------+-------+
|product_id|   category| price|    old_name|   status|amount|base_price|region|priority|user_id|
+----------+-----------+------+------------+---------+------+----------+------+--------+-------+
|      P101|Electronics|299.99|  Item Alpha|Completed|1250.0|     250.0| North|    High|  U1001|
|      P102|  Furniture| 149.5|   Item Beta|  Pending| 450.0|     120.0| South|     Low|  U1002|
|      P103|Electronics| 899.0|  Item Gamma|Completed|3200.5|     750.0|  East|    High|   NULL|
|      P104|   Clo

## Q4: CSV vs. Parquet Storage Formats

### Storage Comparison
- **CSV (Row-Based, Text)**: Data is stored line-by-line as plain text. To extract values for a specific column, Spark must parse every line from beginning to end. It does not support native schema metadata or dictionary compression.
- **Parquet (Columnar, Binary)**: Data is partitioned by columns into row groups. Homogeneous data within columns is compressed using techniques like Run-Length Encoding (RLE) and Snappy compression. File footers contain min/max statistics for every block.

### Performance Impact
1. **Column Pruning**: Queries selecting 2 columns out of 50 only read the exact bytes for those 2 columns from disk, skipping 96% of I/O.
2. **Compression Ratio**: Storing identical column data types together results in much smaller file sizes compared to text CSV.
3. **Filter Efficiency**: Parquet row group statistics allow Spark to bypass reading entire data blocks when query predicates lie outside min/max bounds.


## Q5: Selecting Product ID and Price for Electronics

### Question
Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the `category` is `'Electronics'`.

### Implementation


In [5]:
df_electronics = df_csv.filter(F.col("category") == "Electronics").select("product_id", "price")
df_electronics.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      P101| 299.99|
|      P103|  899.0|
|      P105|  120.0|
|      P107|  450.0|
|      P109|1299.99|
+----------+-------+



## Q6: Renaming Columns and Type Casting

### Question
Write the code to "revise" a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

### Implementation


In [6]:
df_revised = df_csv.withColumnRenamed("old_name", "new_name") \
    .withColumn("price", F.col("price").cast("double"))

df_revised.printSchema()
df_revised.select("product_id", "new_name", "price").show(5)

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- new_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------+------------+------+
|product_id|    new_name| price|
+----------+------------+------+
|      P101|  Item Alpha|299.99|
|      P102|   Item Beta| 149.5|
|      P103|  Item Gamma| 899.0|
|      P104|  Item Delta|  45.0|
|      P105|Item Epsilon| 120.0|
+----------+------------+------+
only showing top 5 rows


## Q7: Fault Tolerance via Lineage Graph (DAG)

### Explanation
Spark DataFrames and RDDs are **immutable**. Instead of continuously copying or snapshotting data across cluster nodes for fault tolerance, Spark tracks every transformation step applied to a dataset inside a Directed Acyclic Graph (DAG) known as the **Lineage Graph**.

When a worker node fails and a data partition in memory is lost:
1. The Driver detects which worker and partition failed.
2. It looks up the DAG lineage graph for that specific partition.
3. The Driver re-executes *only* the missing partition's transformation steps from the nearest available data source or cached checkpoint.
4. Unaffected partitions on healthy worker nodes continue running uninterrupted without restarting the entire job.


## Q8: Multi-Condition Filtering (AND Logic)

### Question
Write a query to filter a DataFrame `df_orders` for rows where the `status` is `'Completed'` AND the `amount` is greater than `1000`.

### Implementation


In [7]:
df_orders = df_csv
df_filtered_orders = df_orders.filter((F.col("status") == "Completed") & (F.col("amount") > 1000))
df_filtered_orders.show()

+----------+---------------+-------+----------+---------+------+----------+------+--------+-------+
|product_id|       category|  price|  old_name|   status|amount|base_price|region|priority|user_id|
+----------+---------------+-------+----------+---------+------+----------+------+--------+-------+
|      P101|    Electronics| 299.99|Item Alpha|Completed|1250.0|     250.0| North|    High|  U1001|
|      P103|    Electronics|  899.0|Item Gamma|Completed|3200.5|     750.0|  East|    High|   NULL|
|      P106|Office Supplies|  15.99| Item Zeta|Completed|2200.0|      12.5| North|     Low|  U1006|
|      P109|    Electronics|1299.99| Item Iota|Completed|1450.0|    1100.0| North|  Medium|  U1009|
+----------+---------------+-------+----------+---------+------+----------+------+--------+-------+



## Q9: Predicate Pushdown in Parquet

### Explanation
**Predicate Pushdown** is an optimization where filter conditions (`WHERE` / `.filter()` clauses) are evaluated directly at the storage level before data is deserialized into Spark Executor memory.

Parquet files contain metadata footers storing minimum and maximum values for each column within each Row Group.

When Spark reads a Parquet file with a filter like `.filter(F.col("amount") > 1000)`:
1. The Parquet reader inspects the footer metadata of each Row Group.
2. If a Row Group's `max(amount)` is `850`, Spark skips reading that entire Row Group from disk.
3. Only Row Groups whose min/max range overlaps with the predicate are loaded into memory, drastically reducing network I/O, disk operations, and memory consumption.


## Q10: Adding Derived Columns (Tax Calculation)

### Question
Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by `1.18` (18% tax).

### Implementation


In [8]:
df_taxed = df_csv.withColumn("final_price", F.round(F.col("base_price") * 1.18, 2))
df_taxed.select("product_id", "base_price", "final_price").show(5)

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|      P101|     250.0|      295.0|
|      P102|     120.0|      141.6|
|      P103|     750.0|      885.0|
|      P104|      35.0|       41.3|
|      P105|     100.0|      118.0|
+----------+----------+-----------+
only showing top 5 rows


## Q11: Transformations vs. Actions

### Comparison
| Feature | Transformations | Actions |
|---|---|---|
| **Definition** | Functions that construct a new DataFrame from an existing one | Functions that evaluate data and return a result to Driver or write to storage |
| **Execution** | **Lazy** (updates DAG logical plan only) | **Eager** (triggers cluster execution) |
| **Return Type** | New `DataFrame` / `Dataset` | Concrete values (`List`, `Long`, unit file output) |

### Examples
- **Transformations**: `.filter()`, `.select()`, `.withColumn()`, `.groupBy()`, `.join()`
- **Actions**: `.show()`, `.collect()`, `.count()`, `.write.csv()`, `.take()`


## Q12: End-to-End Pipeline (Read Parquet -> Filter Nulls -> Write CSV)

### Question
Write the Spark command to load a Parquet file from `"path/to/input"`, filter out any rows where `user_id` is null, and save the result as a CSV at `"path/to/output"`.

### Implementation


In [9]:
df_parquet_in = spark.read.parquet("data/input_parquet/data.parquet")
df_filtered_users = df_parquet_in.filter(F.col("user_id").isNotNull())

df_filtered_users.show()
df_filtered_users.toPandas().to_csv("output/filtered_users.csv", index=False)
print("Filtered dataset successfully written to output/filtered_users.csv")

+-------+----------+------+------+---------+
|user_id|product_id|amount|region|   status|
+-------+----------+------+------+---------+
|  U1001|      P101|1250.0| North|Completed|
|  U1002|      P102| 450.0| South|  Pending|
|  U1004|      P104|  80.0| North|Completed|
|  U1005|      P105|1500.0|  West|Cancelled|
|  U1007|      P107| 950.0| South|Completed|
|  U1008|      P108|1800.0|  West|  Pending|
|  U1010|      P110| 600.0|  East|Completed|
+-------+----------+------+------+---------+

Filtered dataset successfully written to output/filtered_users.csv


## Q13: Client Mode vs. Cluster Mode

### Architectural Differences
- **Client Mode**:
  - The **Driver process** runs on the client machine where the application was submitted (e.g., local laptop or edge workstation).
  - Executors run inside worker nodes across the cluster.
  - **Best For**: Interactive work, debugging, and Jupyter Notebook sessions where output needs to be viewed immediately on the submission machine.
- **Cluster Mode**:
  - The client machine submits the application request to the Cluster Manager and can disconnect.
  - The Cluster Manager selects a worker node inside the cluster and launches the **Driver process** inside an Application Master container.
  - **Best For**: Production batch jobs scheduled via tools like Apache Airflow, preventing job failure if a local workstation disconnects.


## Q14: Multi-Condition Filtering (OR Logic)

### Question
Write a query to filter a dataset for rows where the `region` is `'North'` OR the `priority` is `'High'`.

### Implementation


In [10]:
df_north_or_high = df_csv.filter((F.col("region") == "North") | (F.col("priority") == "High"))
df_north_or_high.show()

+----------+---------------+-------+----------+---------+------+----------+------+--------+-------+
|product_id|       category|  price|  old_name|   status|amount|base_price|region|priority|user_id|
+----------+---------------+-------+----------+---------+------+----------+------+--------+-------+
|      P101|    Electronics| 299.99|Item Alpha|Completed|1250.0|     250.0| North|    High|  U1001|
|      P103|    Electronics|  899.0|Item Gamma|Completed|3200.5|     750.0|  East|    High|   NULL|
|      P104|       Clothing|   45.0|Item Delta|Completed|  80.0|      35.0| North|  Medium|  U1004|
|      P106|Office Supplies|  15.99| Item Zeta|Completed|2200.0|      12.5| North|     Low|  U1006|
|      P107|    Electronics|  450.0|  Item Eta|Completed| 950.0|     380.0| South|    High|  U1007|
|      P108|      Furniture|  620.0|Item Theta|  Pending|1800.0|     500.0|  West|    High|   NULL|
|      P109|    Electronics|1299.99| Item Iota|Completed|1450.0|    1100.0| North|  Medium|  U1009|


## Q15: Memory Safety: `.show(5)` vs. `.collect()`

### Explanation
- **`.collect()`**: Fetches **every row** from all distributed partitions across all executor worker nodes and pulls them over the network into a single array in the Driver process's RAM. On multi-terabyte datasets, this immediately causes an `OutOfMemoryError` (OOM crash) on the Driver node.
- **`.show(5)`**: Requests only the first 5 records from the initial active partition, formats them into a clean ASCII table, and sends only those 5 records to the Driver. It consumes negligible memory and executes almost instantly regardless of total dataset size.


## Superstore Sales Data Pipeline

Below is a complete PySpark workflow utilizing the uploaded Superstore dataset (`Sample - Superstore.csv`) to compute category level sales metrics.


In [11]:
df_superstore = spark.read.csv("data/Sample - Superstore.csv", header=True, inferSchema=True, escape='"')

print("Superstore Dataset Schema:")
df_superstore.printSchema()

print("\nCategory & Sub-Category Performance Metrics:")
superstore_metrics = df_superstore.groupBy("Category", "Sub-Category") \
    .agg(
        F.round(F.sum("Sales"), 2).alias("Total_Sales"),
        F.round(F.avg("Sales"), 2).alias("Avg_Sales"),
        F.count("Order ID").alias("Total_Orders")
    ).orderBy(F.desc("Total_Sales"))

superstore_metrics.show(10, truncate=False)
superstore_metrics.toPandas().to_csv("output/superstore_category_summary.csv", index=False)
print("Saved summary metrics to output/superstore_category_summary.csv")

Superstore Dataset Schema:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)


Category & Sub-Category Performance Metrics:
+---------------+------------+-----------+---------+------------+
|Cate